# kiji-inspector quickstart — Gemma 4 E4B SAEs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataiku/kiji-inspector/blob/main/demo/quickstart_colab.ipynb)

Interpret a residual-stream activation of `google/gemma-4-E4B-it` with the published
[layer-sweep SAEs](https://huggingface.co/575-lab/kiji-inspector-google-gemma-4-E4B-it)
(layers 18/24/30/36).

> **Hardware:** the model loads in bf16 (~16 GB of weights), so a 24 GB GPU such as a Colab
> **L4** or A100 works. A free-tier T4 (15 GB) is too small.

In [ ]:
!pip install -U -q kiji-inspector transformers accelerate

## Build the agent prompt and extract the decision-token activation

The SAEs were trained on activations at the **decision token**: an agent system prompt with a
tool list, a user request, and the assistant prefill `"I'll use the"` — the last token is where
the model commits to a tool choice. We reproduce that format exactly, including the prefill's
missing trailing space (a trailing space is folded into the next token by the tokenizer, which
puts the prompt off-distribution).

> **Layer indexing:** SAE `layer_30` is the residual stream *entering* decoder layer 30 —
> `hidden_states[30]` in HF's `output_hidden_states` tuple, equivalently the output of
> `layers[29]`. A forward **pre-hook** on `layers[30]` captures exactly that, matching
> `kiji_inspector.extraction.activation_extractor`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

MODEL_ID = "google/gemma-4-E4B-it"
SAE_REPO = "575-lab/kiji-inspector-google-gemma-4-E4B-it"
SAE_LAYER = 30  # residual stream entering decoder layer 30
PREFILL = "I'll use the"  # no trailing space — see note above

# The tool_selection scenario the SAEs were trained on (scenarios/tool_selection.json).
SYSTEM_PROMPT = "You are a helpful assistant. Choose the best tool for each request."

TOOLS = [
    ("internal_search", "Search internal company documentation"),
    ("web_search", "Search the public web"),
    ("file_read", "Read a local file"),
    ("file_write", "Write or update a local file"),
    ("database_query", "Query a SQL database"),
    ("api_call", "Call an external REST API"),
    ("code_execute", "Execute code in a sandbox"),
    ("delegate_agent", "Delegate to a sub-agent for complex tasks"),
]

USER_REQUEST = "What is our company's incident response plan as outlined in internal security docs?"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map="auto", low_cpu_mem_usage=True
)
model.eval()

layers = model.model.language_model.layers
input_device = model.model.language_model.embed_tokens.weight.device
print(f"{len(layers)} decoder layers | pre-hooking layers[{SAE_LAYER}] for SAE layer_{SAE_LAYER}")

In [ ]:
def build_decision_prompt(user_request: str) -> str:
    """Mirror kiji_inspector.extraction.extractor.build_agent_prompt."""
    tool_descriptions = "\n".join(f"- {name}: {desc}" for name, desc in TOOLS)
    messages = [
        {
            "role": "system",
            "content": (
                f"{SYSTEM_PROMPT}\n\n"
                f"Available tools:\n{tool_descriptions}\n\n"
                f"When you decide to use a tool, respond with the tool name."
            ),
        },
        {"role": "user", "content": user_request},
    ]
    formatted = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return formatted + PREFILL


def decision_token_activation(user_request: str) -> torch.Tensor:
    """Residual stream entering layers[SAE_LAYER] at the last (decision) token."""
    text = build_decision_prompt(user_request)
    # The chat template already emits <bos>, so don't let the tokenizer add another.
    inputs = processor.tokenizer(text, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    captured = {}

    def pre_hook(_module, args, kwargs):
        captured["h"] = (args[0] if args else kwargs["hidden_states"]).detach()

    handle = layers[SAE_LAYER].register_forward_pre_hook(pre_hook, with_kwargs=True)
    try:
        with torch.inference_mode():
            model(**inputs, use_cache=False)
    finally:
        handle.remove()
    return captured["h"][0, -1].float().cpu()


activation = decision_token_activation(USER_REQUEST)
activation.shape

## Load the SAE and describe the active features

The SAE was trained on **normalized** activations — `(x - mean_vec) / rms_scale`, a single
global mean vector and RMS constant computed over the training set (not per-vector
normalization). `sae.normalize_input` applies that transform; without it the JumpReLU
thresholds are meaningless and the reported features are noise. `sae.denormalize_output`
inverts it, mapping a reconstruction back to raw activation space.

Requires `kiji-inspector >= 0.6.0`.

In [ ]:
from kiji_inspector import SAE

sae, feature_descriptions = SAE.from_pretrained(repo_id=SAE_REPO, layer=SAE_LAYER)
print(
    f"d_model={sae.d_model} d_sae={sae.d_sae} rms_scale={sae.rms_scale:.4f} "
    f"centered={sae.mean_vec is not None} labeled_features={len(feature_descriptions)}"
)

normalized = sae.normalize_input(activation)
results = sae.describe(normalized, feature_descriptions, top_k=5)

In [ ]:
print(f'"{USER_REQUEST}"\n')

for feature_id, desc, act in results:
    print(f"Feature {feature_id} | activation {act:.2f}")
    if isinstance(desc, dict):
        print(f"  Label:       {desc.get('label', 'N/A')}")
        print(f"  Description: {desc.get('description', 'N/A')}")
        print(f"  Confidence:  {desc.get('confidence', 'N/A')}")
    else:
        print(f"  Label:       {desc}")
    print("-" * 60)

## Encode and reconstruct

`encode`/`decode` operate in normalized space, so raw activations go through
`normalize_input` on the way in and `denormalize_output` on the way out. This round trip is
what an ablation does when it writes a reconstruction back into the model's residual stream —
skipping `denormalize_output` would shift the hidden state by the full activation offset.

In [ ]:
x = activation

features = sae.encode(sae.normalize_input(x))
reconstruction = sae.denormalize_output(sae.decode(features))

cosine = torch.nn.functional.cosine_similarity(x, reconstruction, dim=0).item()
rel_error = (torch.norm(x - reconstruction) / torch.norm(x)).item()
print(f"active features (L0): {(features > 0).sum().item()} / {sae.d_sae}")
print(f"reconstruction cosine similarity: {cosine:.4f}")
print(f"relative reconstruction error:    {rel_error:.4f}")